# Alignment, RLHF/DPO & Fine-Tuning Data Preparation

**Track:** Model Development  
**Toolchain:** HuggingFace Hub · DPO/ORPO · TRL · Label Studio  
**Objective:** Learn how to prepare data for fine-tuning, align models to follow instructions using DPO/ORPO, and navigate the open-source model ecosystem.

---

## 🧠 Why Alignment After Model Development?

In the previous notebook, you learned HOW to fine-tune with LoRA/QLoRA. But we skipped two critical questions:
1. **What data do I fine-tune on?** (Data preparation)
2. **How do I make the model actually helpful?** (Alignment)

### The Alignment Problem

A raw pre-trained model (GPT, Llama, etc.) is like a very smart person who **doesn't know social norms:**
- It can answer questions, but also generates toxic content
- It follows instructions, but also follows harmful ones
- It knows facts, but also confidently states falsehoods

**Alignment** teaches the model to be helpful, harmless, and honest.

### The LLM Training Pipeline

```
Stage 1: Pre-training          → Learn language (trillions of tokens)
         Cost: $1M-$100M       → Result: Smart but uncontrolled model
                                  (You don't do this — companies like Meta do)

Stage 2: Supervised Fine-Tuning (SFT) → Learn to follow instructions
         Cost: $100-$10K       → Result: Helpful model
                                  (LoRA/QLoRA from previous notebook)

Stage 3: Alignment (RLHF/DPO)  → Learn human preferences
         Cost: $100-$5K        → Result: Helpful + Safe model
                                  (THIS notebook)
```

**🤔 Can I skip alignment?** Yes, if your use case is narrow and controlled (e.g., internal code assistant). But for any user-facing application, alignment is essential.

---

## 📑 Table of Contents

* [🧠 Why Alignment After Model Development?](#why-alignment-after-model-development)
  * [The Alignment Problem](#the-alignment-problem)
  * [The LLM Training Pipeline](#the-llm-training-pipeline)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Navigating the Open-Source Model Ecosystem](#1-navigating-the-open-source-model-ecosystem)
  * [🤔 Where Do I Find Models?](#where-do-i-find-models)
  * [Common Licenses for Open Models](#common-licenses-for-open-models)
  * [🤔 How to Choose a Base Model?](#how-to-choose-a-base-model)
* [2 · Preparing Fine-Tuning Data](#2-preparing-fine-tuning-data)
  * [🤔 What Does Good Fine-Tuning Data Look Like?](#what-does-good-fine-tuning-data-look-like)
  * [Quality Over Quantity](#quality-over-quantity)
  * [Data Labeling Strategies](#data-labeling-strategies)
* [3 · RLHF vs DPO vs ORPO: Alignment Methods](#3-rlhf-vs-dpo-vs-orpo-alignment-methods)
  * [🤔 What is Alignment?](#what-is-alignment)
  * [The Evolution of Alignment](#the-evolution-of-alignment)
  * [🤔 How Does DPO Work? (Simplified)](#how-does-dpo-work-simplified)
  * [⚖️ Which Alignment Method Should I Use?](#which-alignment-method-should-i-use)
* [4 · Data Annotation at Scale](#4-data-annotation-at-scale)
  * [🤔 How Do Real Companies Build Training Data?](#how-do-real-companies-build-training-data)
  * [The Annotation Pipeline](#the-annotation-pipeline)
* [Knowledge Check](#knowledge-check)
  * [Question 1: SFT vs DPO](#question-1-sft-vs-dpo)
  * [Question 2: Beta Parameter](#question-2-beta-parameter)
  * [Question 3: Preference Data Quality](#question-3-preference-data-quality)
  * [Question 4: License Check](#question-4-license-check)
* [Practical Practice](#practical-practice)
  * [Exercise 1: Preference Pair Writing](#exercise-1-preference-pair-writing)
  * [Exercise 2: DPO Loss Calculation](#exercise-2-dpo-loss-calculation)
  * [Exercise 3: Model Selection](#exercise-3-model-selection)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Questions](#key-questions)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Models predict the *next token*; they do not inherently know how to be a 'helpful assistant'. Seniors use alignment techniques (DPO/RLHF) so models learn human preferences, nuance, and desired output formats.

**Mental Model:** Supervised Fine-Tuning (SFT) teaches the model to *speak the language* of the task. Direct Preference Optimization (DPO) teaches the model *which answers are polite/helpful and which are toxic*.

**Common Junior Pitfall:** Assuming fine-tuning fixes base model idiocy. If the base model doesn't contain the knowledge, alignment won't magically inject facts; it only changes the *style* of the response.

---


## 1 · Navigating the Open-Source Model Ecosystem

### 🤔 Where Do I Find Models?

**HuggingFace Hub** is the GitHub of AI models. It hosts 500,000+ models. Here's how to navigate it:

| What to Check | Where to Find It | Why It Matters |
|--------------|-----------------|----------------|
| **Model Card** | Model page → README | Architecture, training data, limitations |
| **License** | Model page → License tab | Can you use it commercially? |
| **Leaderboard scores** | Open LLM Leaderboard | Benchmark comparisons |
| **VRAM requirement** | Model Card or community | Will it fit on your GPU? |

### Common Licenses for Open Models

| License | Commercial Use | Modify? | Example Models |
|---------|---------------|---------|----------------|
| **Apache 2.0** | ✅ Yes | ✅ Yes | Mistral, Gemma |
| **MIT** | ✅ Yes | ✅ Yes | Some research models |
| **Llama 3 Community** | ✅ Yes (< 700M users) | ✅ Yes | Llama 3, Llama 3.1 |
| **CC-BY-NC** | ❌ No | ✅ Yes | Academic models only |
| **Custom/Restricted** | ❓ Read carefully | ❓ Varies | Phi, some research |

**⚠️ Always check the license BEFORE fine-tuning!** Using a non-commercial model in a product is a legal risk.

### 🤔 How to Choose a Base Model?

| Model Size | VRAM Needed (fp16) | VRAM Needed (4-bit) | Best For |
|-----------|-------------------|--------------------|---------|
| 1-3B | 4-6 GB | 2-3 GB | Edge, mobile, simple tasks |
| 7-8B | 14-16 GB | 4-6 GB | Best quality-to-cost ratio |
| 13B | 26 GB | 8-10 GB | Better reasoning |
| 34-40B | 68-80 GB | 20-24 GB | Near-GPT-4 quality |
| 70B | 140 GB | 40 GB | State-of-the-art open |

**Rule of thumb:** Start with a 7-8B model. Only go bigger if quality is insufficient.

In [ ]:
# Cell 1 — Model selection helper

popular_models_2026 = [
    {'name': 'Llama 3.3 8B',    'params': 8,   'license': 'Llama Community', 'commercial': True,  'vram_fp16': 16, 'vram_4bit': 5,  'strengths': 'Best overall 8B, multilingual'},
    {'name': 'Llama 3.3 70B',   'params': 70,  'license': 'Llama Community', 'commercial': True,  'vram_fp16': 140,'vram_4bit': 40, 'strengths': 'Best open source large'},
    {'name': 'Mistral 7B v0.3', 'params': 7,   'license': 'Apache 2.0',      'commercial': True,  'vram_fp16': 14, 'vram_4bit': 4,  'strengths': 'Fast, great for fine-tuning'},
    {'name': 'Mixtral 8x7B',    'params': 47,  'license': 'Apache 2.0',      'commercial': True,  'vram_fp16': 94, 'vram_4bit': 26, 'strengths': 'MoE: 47B params, 13B active'},
    {'name': 'Gemma 2 9B',      'params': 9,   'license': 'Apache 2.0',      'commercial': True,  'vram_fp16': 18, 'vram_4bit': 6,  'strengths': 'Google, excellent for size'},
    {'name': 'Phi-3 Mini 3.8B', 'params': 3.8, 'license': 'MIT',             'commercial': True,  'vram_fp16': 8,  'vram_4bit': 3,  'strengths': 'Best tiny model, edge/mobile'},
    {'name': 'Qwen 2.5 7B',     'params': 7,   'license': 'Apache 2.0',      'commercial': True,  'vram_fp16': 14, 'vram_4bit': 4,  'strengths': 'Great for code and math'},
]

def recommend_model(task, max_vram_gb, need_commercial=True):
    """Recommend a model based on constraints."""
    candidates = [m for m in popular_models_2026 
                  if m['vram_4bit'] <= max_vram_gb 
                  and (not need_commercial or m['commercial'])]
    return sorted(candidates, key=lambda x: x['params'], reverse=True)

print('🔍 Model Recommendations')
print('=' * 70)

# Scenario 1: Consumer GPU (16GB)
print('\n📱 Scenario: Consumer GPU (16GB VRAM, commercial use)')
for m in recommend_model('general', 16, True)[:3]:
    print(f'  ✅ {m["name"]} — {m["vram_4bit"]}GB (4-bit) — {m["strengths"]}')

# Scenario 2: Cloud A100 (80GB)
print('\n☁️ Scenario: Cloud A100 (80GB VRAM, commercial use)')
for m in recommend_model('general', 80, True)[:3]:
    print(f'  ✅ {m["name"]} — {m["vram_4bit"]}GB (4-bit) — {m["strengths"]}')

---

## 2 · Preparing Fine-Tuning Data

### 🤔 What Does Good Fine-Tuning Data Look Like?

There are three common data formats:

| Format | Structure | Used For |
|--------|----------|----------|
| **Instruction-tuning** | `{instruction, input, output}` | Teaching the model to follow instructions |
| **Conversational** | `[{role, content}, ...]` | Multi-turn chat behavior |
| **Preference pairs** | `{prompt, chosen, rejected}` | DPO/RLHF alignment |

### Quality Over Quantity

| Dataset Size | Quality | Result |
|-------------|---------|--------|
| 100K low-quality | Noisy, contradictory | Model gets confused |
| 1K high-quality | Clean, consistent, diverse | Model performs well |
| 10K high-quality | Gold standard | Excellent performance |

**Key insight:** 1,000 expertly curated examples often beat 100,000 scraped examples.

### Data Labeling Strategies

| Strategy | Cost | Quality | Best For |
|----------|------|---------|----------|
| **Manual expert labeling** | High ($20-50/hr) | Highest | Medical, legal, safety-critical |
| **Crowdsourced** (Scale AI, Surge) | Medium ($5-15/hr) | Medium-High | General instruction data |
| **LLM-assisted** (GPT-4 generates, human validates) | Low | Medium | Bootstrapping initial data |
| **Self-instruct** (model generates its own data) | Very low | Lower | Early experimentation |

**⚠️ The LLM-assisted trap:** Using GPT-4 to generate training data for your model may violate the provider's Terms of Service. Always check.

In [ ]:
# Cell 2 — Build instruction-tuning and preference datasets
import json

# FORMAT 1: Instruction-tuning (SFT)
sft_data = [
    {
        'instruction': 'Summarize this customer complaint in one sentence.',
        'input': 'I ordered a laptop 3 weeks ago and it still has not arrived. I called support twice and they said it\'s "in transit" but the tracking hasn\'t updated in 10 days. I want a refund.',
        'output': 'Customer is requesting a refund for a laptop ordered 3 weeks ago that shows no shipping progress despite contacting support twice.'
    },
    {
        'instruction': 'Classify the sentiment of this product review.',
        'input': 'The battery life is amazing but the screen is dim outdoors. Overall decent for the price.',
        'output': '{"sentiment": "mixed_positive", "pros": ["battery life"], "cons": ["screen brightness"], "recommendation": "conditional_buy"}'
    },
]

# FORMAT 2: Conversational (chat fine-tuning)
chat_data = [
    {
        'messages': [
            {'role': 'system', 'content': 'You are an internal HR assistant. Only answer questions about company policies.'},
            {'role': 'user', 'content': 'How many vacation days do I get?'},
            {'role': 'assistant', 'content': 'Full-time employees receive 20 vacation days per year, accrued monthly at 1.67 days/month. Part-time employees receive a pro-rated amount.'},
            {'role': 'user', 'content': 'Can I carry unused days to next year?'},
            {'role': 'assistant', 'content': 'Yes, you can carry up to 5 unused days to the next calendar year. Days beyond 5 expire on December 31st. Please submit carryover requests through the HR portal by November 30th.'}
        ]
    }
]

# FORMAT 3: Preference pairs (for DPO/RLHF)
preference_data = [
    {
        'prompt': 'Explain what a database index is.',
        'chosen': 'A database index is like a book\'s table of contents. Instead of reading every page to find a topic, you look up the page number in the index. Similarly, a database index stores pointers to rows so the database can find data without scanning every row. This makes queries much faster, especially on large tables.',
        'rejected': 'A database index is a data structure that improves the speed of data retrieval operations on a database table at the cost of additional writes and storage space to maintain the index data structure. Indexes are used to quickly locate data without having to search every row.'
    }
]

print('📦 Fine-Tuning Data Formats')
print('=' * 60)
print('\n1️⃣ Instruction-Tuning (SFT):')
print(f'   Fields: instruction, input, output')
print(f'   Example: {json.dumps(sft_data[0], indent=2)[:200]}...')
print(f'\n2️⃣ Conversational (Chat):')
print(f'   Fields: messages[{{role, content}}]')
print(f'   Turns: {len(chat_data[0]["messages"])} messages')
print(f'\n3️⃣ Preference Pairs (DPO/RLHF):')
print(f'   Fields: prompt, chosen (good), rejected (bad)')
print(f'   Chosen length: {len(preference_data[0]["chosen"])} chars')
print(f'   Rejected length: {len(preference_data[0]["rejected"])} chars')
print(f'\n💡 Notice: The "chosen" response is friendlier and uses an analogy.')
print(f'   The "rejected" response is technically correct but dry and unhelpful.')
print(f'   DPO teaches the model to prefer the "chosen" style.')

---

## 3 · RLHF vs DPO vs ORPO: Alignment Methods

### 🤔 What is Alignment?

After SFT, the model can follow instructions. But it doesn't know which response STYLE humans prefer. Alignment teaches preferences.

### The Evolution of Alignment

| Method | Year | How It Works | Complexity | Cost |
|--------|------|-------------|------------|------|
| **RLHF** | 2022 | Train a reward model → Use PPO to optimize | Very high | Very high |
| **DPO** | 2023 | Directly optimize on preference pairs | Medium | Medium |
| **ORPO** | 2024 | Combine SFT + alignment in one step | Low | Low |
| **SimPO** | 2024 | Simplified DPO without reference model | Low | Low |

### 🤔 How Does DPO Work? (Simplified)

1. You give the model a prompt
2. You show it two responses: one GOOD (chosen), one BAD (rejected)
3. DPO adjusts the model weights so it's MORE likely to generate the good response and LESS likely to generate the bad one

```
Prompt: "Explain machine learning"

Chosen (good):  "Machine learning is like teaching a child by showing examples..."
Rejected (bad): "ML is a subset of AI that uses statistical methods to..."

→ DPO increases P(chosen) and decreases P(rejected)
```

### ⚖️ Which Alignment Method Should I Use?

| If you have... | Use | Why |
|----------------|-----|-----|
| < 1K preference pairs | **ORPO** | Combines SFT + alignment, works with less data |
| 1K-10K preference pairs | **DPO** | Best balance of quality and simplicity |
| 10K+ pairs + large budget | **RLHF** | Highest quality but complex |
| No preference data yet | **Start with SFT** | Alignment comes AFTER basic fine-tuning |

In [ ]:
# Cell 3 — Simulate DPO training
import numpy as np
import json

class SimpleDPOTrainer:
    """Educational simulation of DPO training.
    
    Real DPO uses the model's own probabilities. This simulation
    shows the CONCEPT: adjusting preferences toward chosen responses.
    """
    def __init__(self, beta=0.1):
        self.beta = beta  # controls how strongly to push preferences
        self.preference_scores = {}  # tracks preference shifts
    
    def compute_dpo_loss(self, chosen_score, rejected_score):
        """DPO loss: -log(sigmoid(beta * (chosen - rejected)))"""
        diff = self.beta * (chosen_score - rejected_score)
        loss = -np.log(1 / (1 + np.exp(-diff)))
        return loss
    
    def train_step(self, prompt, chosen, rejected):
        """One DPO training step on a preference pair."""
        # Simulate model scores (in reality, these come from log-probabilities)
        chosen_score = len(chosen) * 0.01 + np.random.normal(0, 0.1)
        rejected_score = len(rejected) * 0.01 + np.random.normal(0, 0.1)
        
        loss = self.compute_dpo_loss(chosen_score, rejected_score)
        
        # After training, the model shifts preference toward chosen
        preference_shift = max(0, chosen_score - rejected_score)
        self.preference_scores[prompt[:30]] = preference_shift
        
        return loss

# Create preference dataset
preference_pairs = [
    {
        'prompt': 'How do I fix a memory leak in Python?',
        'chosen': 'Great question! Memory leaks in Python often come from circular references or unclosed resources. Here\'s a step-by-step debugging approach: 1) Use tracemalloc to find the leak, 2) Check for circular references with gc.get_referrers(), 3) Use context managers (with statements) for resources.',
        'rejected': 'Use gc.collect() to fix memory leaks.'
    },
    {
        'prompt': 'What\'s the difference between REST and GraphQL?',
        'chosen': 'Think of REST like a restaurant with a fixed menu — you pick from predefined endpoints. GraphQL is like a build-your-own bowl — you specify exactly what data you want. REST is simpler but can over-fetch data. GraphQL is flexible but adds complexity.',
        'rejected': 'REST uses multiple endpoints while GraphQL uses a single endpoint with queries.'
    },
    {
        'prompt': 'Explain Docker to a beginner.',
        'chosen': 'Imagine you\'re moving to a new apartment. Instead of packing each item individually and hoping nothing breaks, Docker lets you put your entire room — furniture, decorations, everything — into a shipping container. The container looks the same no matter where it goes. That\'s Docker: your app + all its dependencies in one portable package.',
        'rejected': 'Docker is a containerization platform that packages applications and their dependencies into containers for consistent deployment across environments.'
    },
]

# Train DPO
trainer = SimpleDPOTrainer(beta=0.1)
print('🎯 DPO Training Simulation')
print('=' * 60)

for epoch in range(3):
    epoch_loss = 0
    for pair in preference_pairs:
        loss = trainer.train_step(pair['prompt'], pair['chosen'], pair['rejected'])
        epoch_loss += loss
    print(f'  Epoch {epoch+1}: avg_loss = {epoch_loss/len(preference_pairs):.4f}')

print(f'\n📊 Preference shifts (higher = model prefers chosen more):')
for prompt, shift in trainer.preference_scores.items():
    print(f'  "{prompt}..." → shift: {shift:.3f}')

print(f'\n💡 Key insight: DPO teaches the model to prefer:')
print(f'   ✅ Detailed, step-by-step explanations')
print(f'   ✅ Analogies and beginner-friendly language')
print(f'   ❌ Over terse, jargon-heavy responses')

In [ ]:
# Cell 4 — TRL DPO training script (production reference)
#
# This is the ACTUAL code you'd run to DPO-train a model with HuggingFace TRL.
# It requires a GPU, so we print the script rather than executing it.

dpo_script = '''
# Production DPO Training Script with TRL
# Run with: python dpo_train.py
# Requires: pip install trl transformers peft bitsandbytes datasets

from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from datasets import load_dataset
import torch

# 1. Load base model with QLoRA (4-bit quantization)
model_name = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 2. LoRA configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# 3. Load preference dataset
# Format: {prompt: str, chosen: str, rejected: str}
dataset = load_dataset("your-org/preference-data", split="train")

# 4. DPO training configuration
training_args = DPOConfig(
    output_dir="./dpo-llama3-8b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,             # DPO-specific: controls preference strength
    logging_steps=10,
    save_steps=100,
    bf16=True,
    report_to="wandb",
)

# 5. Train!
trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
)

trainer.train()
trainer.save_model("./dpo-llama3-8b-final")
'''

print('🚀 Production DPO Training Script:')
print(dpo_script)
print('Key parameters:')
print('  beta=0.1  → Lower = gentler preference (default: 0.1)')
print('  r=16      → LoRA rank (from previous notebook)')
print('  epochs=3  → DPO usually needs fewer epochs than SFT')

---

## 4 · Data Annotation at Scale

### 🤔 How Do Real Companies Build Training Data?

| Tool | Type | Cost | Best For |
|------|------|------|----------|
| **Label Studio** | Open-source annotation | Free (self-hosted) | Small-medium teams |
| **Scale AI** | Managed labeling workforce | $1-50/task | Large-scale, production |
| **Argilla** | Open-source + LLM feedback | Free | RLHF/DPO data collection |
| **Prodigy** | Active-learning annotation | $390 license | NLP annotation |

### The Annotation Pipeline

```
Raw Data → [LLM Pre-labels] → [Human Reviews] → [Quality Check] → Training Data
           (GPT-4 generates       (Expert fixes      (Inter-annotator
            initial labels)        mistakes)           agreement check)
```

**This hybrid approach** (LLM + human) is 5-10x faster than pure human labeling while maintaining quality.

In [ ]:
# Cell 5 — Build an annotation pipeline
import json, random

class AnnotationPipeline:
    """Simulates an LLM-assisted annotation pipeline."""
    
    def __init__(self):
        self.annotated = []
        self.stats = {'llm_correct': 0, 'human_fixed': 0, 'rejected': 0}
    
    def llm_prelabel(self, text, task):
        """Stage 1: LLM generates initial annotation (simulated)."""
        labels = {'positive': 0.7, 'negative': 0.2, 'neutral': 0.1}
        label = random.choices(list(labels.keys()), weights=labels.values())[0]
        confidence = random.uniform(0.6, 0.99)
        return {'text': text, 'llm_label': label, 'confidence': confidence}
    
    def human_review(self, item, correct_label):
        """Stage 2: Human expert reviews and corrects."""
        if item['confidence'] > 0.9 and random.random() > 0.1:
            self.stats['llm_correct'] += 1
            item['final_label'] = item['llm_label']
            item['reviewed'] = 'auto_approved'
        else:
            self.stats['human_fixed'] += 1
            item['final_label'] = correct_label
            item['reviewed'] = 'human_corrected'
        
        self.annotated.append(item)
        return item
    
    def quality_report(self):
        """Stage 3: Quality metrics."""
        total = len(self.annotated)
        agreement = self.stats['llm_correct'] / total if total else 0
        return {
            'total_annotated': total,
            'llm_auto_approved': f"{self.stats['llm_correct']} ({agreement:.0%})",
            'human_corrected': self.stats['human_fixed'],
            'time_saved': f"{agreement:.0%} (vs pure human labeling)",
        }

# Run the pipeline
pipeline = AnnotationPipeline()
random.seed(42)

sample_texts = [
    ('Great product, exceeded expectations!', 'positive'),
    ('Terrible quality, broke after 2 days', 'negative'),
    ('It works as described', 'neutral'),
    ('Absolutely love it, best purchase ever', 'positive'),
    ('Not bad for the price', 'neutral'),
] * 10  # 50 samples

for text, true_label in sample_texts:
    prelabeled = pipeline.llm_prelabel(text, 'sentiment')
    pipeline.human_review(prelabeled, true_label)

report = pipeline.quality_report()
print('📊 Annotation Pipeline Results')
print('=' * 50)
for k, v in report.items():
    print(f'  {k:<25}: {v}')

print(f'\n💡 By using LLM pre-labeling, humans only review {report["human_corrected"]} items')
print(f'   instead of all {report["total_annotated"]}. This is 5-10x faster.')

---
## Knowledge Check

### Question 1: SFT vs DPO
What is the difference between SFT and DPO? Can you do DPO without SFT first?

<details>
<summary>Click to reveal answer</summary>

**SFT (Supervised Fine-Tuning)** teaches the model to follow instructions by training on (instruction, response) pairs. It learns WHAT to say. **DPO (Direct Preference Optimization)** teaches the model which response STYLE humans prefer by training on (prompt, chosen, rejected) triplets. It learns HOW to say it. You typically do SFT first, then DPO. Doing DPO without SFT usually gives poor results because the model doesn't yet know how to follow instructions.
</details>

### Question 2: Beta Parameter
In DPO, what happens if you set `beta=0.01` (very low) vs `beta=1.0` (very high)?

<details>
<summary>Click to reveal answer</summary>

`beta=0.01` (low): The model barely shifts its preferences -- minimal alignment effect, the model stays close to its SFT behavior. `beta=1.0` (high): The model aggressively shifts toward chosen responses, but risks "over-aligning" -- becoming too rigid and losing diversity/naturalness. Default `beta=0.1` is a good starting point.
</details>

### Question 3: Preference Data Quality
You have 50K preference pairs generated entirely by GPT-4. Is this good training data? What are the risks?

<details>
<summary>Click to reveal answer</summary>

Risks: (1) **Terms of Service** -- many providers prohibit using outputs to train competing models. (2) **Model collapse** -- training on AI-generated data can create a feedback loop where the model mimics GPT-4's specific quirks/biases. (3) **Lack of diversity** -- all data reflects one model's style. Better approach: use GPT-4 to pre-label, then have humans review and correct (hybrid pipeline).
</details>

### Question 4: License Check
Your company wants to fine-tune Llama 3.1 70B for a commercial product. What license does it use, and is this allowed?

<details>
<summary>Click to reveal answer</summary>

Llama 3.1 uses the **Llama Community License**. Commercial use is allowed for organizations with fewer than 700 million monthly active users. For most companies this is fine. However, you must include Meta's attribution notice and cannot use the model to train other models that compete with Llama. Always read the full license before deploying.
</details>


---
## Practical Practice

### Exercise 1: Preference Pair Writing
Write 3 high-quality preference pairs for a customer support chatbot. Each pair should have a prompt, a chosen (good) response, and a rejected (bad) response. Focus on tone and helpfulness.

### Exercise 2: DPO Loss Calculation
Given: chosen_score = 0.8, rejected_score = 0.3, beta = 0.1. Calculate the DPO loss manually using the formula: `-log(sigmoid(beta * (chosen - rejected)))`.

### Exercise 3: Model Selection
Your team has a single A100 (80GB VRAM) and needs a model for code generation. Which model from the popular models list would you choose and why? Consider license, quality, and memory.


In [ ]:
# ===== EXERCISE SOLUTIONS =====
import numpy as np

print('Ex 1: Sample Preference Pairs')
pairs = [
    {'prompt': 'My order is late. Where is it?',
     'chosen': 'I\'m sorry about the delay! Let me check your order status right now. Could you share your order number so I can give you an exact update?',
     'rejected': 'Check the tracking number on the website.'},
    {'prompt': 'How do I return a product?',
     'chosen': 'I\'d be happy to help with your return! You have 30 days from delivery. Here\'s how: 1) Log into your account, 2) Go to Order History, 3) Click "Start Return" next to the item. Need me to walk you through it?',
     'rejected': 'See our return policy at /returns.'},
]
for i, p in enumerate(pairs[:2]):
    print(f'\n  Pair {i+1}: "{p["prompt"][:50]}..."')
    print(f'    Chosen:   {p["chosen"][:80]}...')
    print(f'    Rejected: {p["rejected"]}')

print('\nEx 2: DPO Loss')
chosen_score, rejected_score, beta = 0.8, 0.3, 0.1
diff = beta * (chosen_score - rejected_score)
sigmoid = 1 / (1 + np.exp(-diff))
loss = -np.log(sigmoid)
print(f'  diff = {beta} * ({chosen_score} - {rejected_score}) = {diff:.2f}')
print(f'  sigmoid({diff:.2f}) = {sigmoid:.4f}')
print(f'  loss = -log({sigmoid:.4f}) = {loss:.4f}')
print(f'  Low loss = model already prefers chosen (good!)')

print('\nEx 3: Model Selection for Code Gen on A100 80GB')
print('  Best choice: Qwen 2.5 7B (Apache 2.0, 4GB at 4-bit)')
print('  Reasoning: Apache 2.0 = fully commercial, great at code/math,')
print('  tiny VRAM footprint leaves room for large batch sizes.')
print('  Alternative: Llama 3.3 70B at 4-bit (40GB) for max quality.')


---

## 🎯 Summary & Key Takeaways

| Concept | What You Learned | Why It Matters |
|---------|-----------------|----------------|
| HuggingFace Hub | Navigate models, check licenses | Choose the right base model |
| Data Formats | Instruction, conversational, preference | Match format to fine-tuning method |
| RLHF vs DPO vs ORPO | Alignment method trade-offs | Choose the right alignment approach |
| DPO Training | Preference pairs → model alignment | Make models helpful and safe |
| Data Annotation | LLM + human pipeline | Build quality data 5-10x faster |

### 🧠 Key Questions

1. **"Do I always need alignment?"** → No. For narrow, controlled use cases (internal tools), SFT alone may suffice.
2. **"How many preference pairs do I need?"** → Start with 1K high-quality pairs. Scale to 10K if quality isn't sufficient.
3. **"Should I use RLHF or DPO?"** → DPO in 2026. RLHF is more powerful but the complexity rarely justifies it.

**Next →** `17_02_rlhf_alignment.ipynb` — Next Module